# KFRE Risk Estimation on a UK Primary Care CKD Cohort

An end-to-end example of the `kfre` package applied to a public chronic kidney disease
cohort: outcome labelling, unit handling, batch risk estimation for the 4- and
6-variable Kidney Failure Risk Equation, and performance assessment.

### Data

The cohort is the open dataset deposited alongside:

> Major RW, Shepherd D, Medcalf JF, Xu G, Gray LJ, Brunskill NJ.
> *The Kidney Failure Risk Equation for prediction of end stage renal disease in UK
> primary care: An external validation and clinical impact projection cohort study.*
> PLoS Med. 2019;16(11):e1002955. doi:10.1371/journal.pmed.1002955

Dataset DOI: <https://doi.org/10.25392/leicester.data.9860807.v1>

**The data file is not redistributed in this repository.** Download it from the Figshare
DOI above and place it at `../data/kfre_plosmed_v2_0.csv` before running this notebook.

### Scope

Only the 4- and 6-variable KFRE forms are computed. The 8-variable form requires serum
albumin, bicarbonate, calcium, and phosphate, none of which are present in this dataset.

## 1. Environment and Imports

In [ ]:
# !pip install kfre eda_toolkit

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import kfre
from kfre import (
    add_kfre_risk_col,
    bootstrap_metric_ci,
    class_ckd_stages,
    class_esrd_outcome,
    plot_kfre_metrics,
)
from eda_toolkit import generate_table1

print(f"kfre    {kfre.__version__}")
print(f"pandas  {pd.__version__}")

## 2. Load the Cohort

Column meanings follow the S2 Text of the source publication.

| Column | Meaning |
| --- | --- |
| `age` | Age in years at baseline |
| `epi_egfr` | CKD-EPI eGFR, ml/min/1.73 m<sup>2</sup> |
| `female` | 1 = female, 0 = male |
| `acr_mgmmol` | Urine albumin-to-creatinine ratio, **mg/mmol** |
| `esrd` | End stage renal disease event indicator |
| `time` | Follow-up duration in days |
| `death` | All-cause death indicator |
| `dm`, `htn`, `cvd`, `hf` | Diabetes, hypertension, cardiovascular disease, heart failure |
| `neph_known` | Previously seen by secondary care renal services |

In [ ]:
df = pd.read_csv("../data/kfre_plosmed_v2_0.csv").set_index("id")
print(f"DataFrame shape: {df.shape}")
df.head()

## 3. Unit Conversion for the KFRE ACR Term

The KFRE was developed with urine ACR expressed in **mg/g**. This dataset records ACR in
**mg/mmol**, which is the usual reporting unit in the UK and much of Europe. The
conversion factor is 8.8402 mg/g per mg/mmol.

This step matters more than it looks. The equation includes a `ln(ACR)` term with
coefficient 0.4510, so passing mg/mmol into a model expecting mg/g shifts the linear
predictor by

$$0.4510 \times \ln(8.8402) = 0.9829$$

which multiplies $\exp(X\beta)$ by 0.374 and makes every predicted risk roughly 2.7 times
too low. Because the shift is constant, it is a monotone transform of the risk score:
rank-based metrics such as AUC ROC and average precision are unchanged, while Brier
score, calibration, and every absolute risk threshold are wrong. A model that looks fine
on discrimination alone can be badly miscalibrated for this reason, so always confirm the
ACR units before scoring.

In [ ]:
MGMMOL_TO_MGG = 8.8402  # mg/g per mg/mmol
df["acr_mgg"] = df["acr_mgmmol"] * MGMMOL_TO_MGG
df[["acr_mgmmol", "acr_mgg"]].describe().T

## 4. Analytic Cohort Definition

Two exclusions are applied before scoring.

**Age below 18.** The KFRE was developed in adult CKD cohorts.

**ACR of zero.** The equation takes `ln(ACR)`, which is undefined at zero. Values of zero
appear when a laboratory reports an undetectable or negative result. Complete-case
exclusion is the standard handling. Any analysis using this dataset should state the
count explicitly, since these patients differ from the scoreable population.

A helper variable `sex_str` is created because `add_kfre_risk_col` expects a string sex
column.

In [ ]:
df["sex_str"] = df["female"].map({1: "Female", 0: "Male"})

n_before = len(df)
n_under18 = int((df["age"] < 18).sum())
n_zero_acr = int((df["acr_mgmmol"] <= 0).sum())

df = df[(df["age"] >= 18) & (df["acr_mgmmol"] > 0)].copy()

print(f"Deposited cohort:        {n_before:,}")
print(f"  excluded, age < 18:    {n_under18:,}")
print(f"  excluded, ACR = 0:     {n_zero_acr:,}")
print(f"Analytic cohort:         {len(df):,}")

A small number of patients exceed the KFRE's upper development age of 100. They are
retained, and `add_kfre_risk_col` emits an expected out-of-bounds warning for them.

## 5. Outcome Labelling and Censoring Sensitivity

`class_esrd_outcome` builds a binary outcome label for a chosen horizon from a
user-supplied kidney failure event indicator (here `esrd`) and a follow-up duration
column (`time`). It does not derive the event from eGFR.

### The censoring caveat

Fixed-horizon labelling marks every patient without a recorded event as 0, including
those whose follow-up ended before the horizon through death or loss to follow-up. Those
patients are censored, so labelling them as confirmed event-free overstates the number of
known non-events. Median follow-up in this cohort is under 5 years and roughly a quarter
of patients died, so this affects a meaningful share of the 5-year labels.

Setting `censor_incomplete=True` labels such patients `NaN` so they can be excluded,
which tests whether performance is robust to the concern. Complete-case exclusion is a
pragmatic guard, and a full competing-risks model would be the rigorous treatment. The
KFRE itself was derived with censoring-aware Cox models (Tangri et al.), so published
C-statistics for the equation are Harrell's C on censored time-to-event data and are not
directly comparable to fixed-horizon ROC AUC.

In [ ]:
# Primary labelling: fixed horizon, censored patients treated as non-events.
for y in (2, 5):
    df = class_esrd_outcome(
        df.copy(),
        col="esrd",
        years=y,
        duration_col="time",
        prefix="esrd",
        create_years_col=True,
        censor_incomplete=False,
    )
    print(f"{y}-year outcome:\n{df[f'esrd_{y}_year_outcome'].value_counts()}\n")

In [ ]:
# Sensitivity labelling: patients with incomplete follow-up set to NaN.
rows = []
for y in (2, 5):
    tmp = class_esrd_outcome(
        df.copy(),
        col="esrd",
        years=y,
        duration_col="time",
        prefix="esrd_cens",
        create_years_col=True,
        censor_incomplete=True,
    )
    s = tmp[f"esrd_cens_{y}_year_outcome"]
    df[f"esrd_cens_{y}_year_outcome"] = s
    rows.append(
        {
            "horizon": f"{y}-year",
            "events": int((s == 1).sum()),
            "non-events": int((s == 0).sum()),
            "censored (NaN)": int(s.isna().sum()),
            "analysable": int(s.notna().sum()),
        }
    )

pd.DataFrame(rows).set_index("horizon")

## 6. Cohort Descriptives

`generate_table1` from `eda_toolkit` summarises continuous and categorical columns in one
pass. `female` is dropped because it is now represented by `sex_str`.

In [ ]:
df_table1 = generate_table1(df.drop(columns=["female"]), value_counts=True)
df_table1 = df_table1.drop(
    columns=["Mode", "Mean", "SD", "Count", "Proportion (%)", "Type"]
)
df_table1

## 7. Compute KFRE Risk

`add_kfre_risk_col` appends risk columns for every requested model variant and horizon in
a single pass, named `kfre_{n}var_{y}year`.

`is_north_american=False` selects the non-North American baseline survival values
(0.9832 at 2 years, 0.9365 at 5 years). Tangri et al. found baseline risk to be lower
outside North America and proposed this calibration factor, so it is the appropriate
choice for a UK cohort.

In [ ]:
df = add_kfre_risk_col(
    df=df,
    age_col="age",
    sex_col="sex_str",
    eGFR_col="epi_egfr",
    uACR_col="acr_mgg",  # mg/g, converted in Section 3
    dm_col="dm",
    htn_col="htn",
    num_vars=[4, 6],
    years=(2, 5),
    is_north_american=False,
    copy=False,
)

kfre_cols = [c for c in df.columns if c.startswith("kfre_")]
df[kfre_cols].head()

In [ ]:
df[kfre_cols].describe().T

In [ ]:
# Confirm no missing predictions or outcomes before evaluating.
outcome_cols = [c for c in df.columns if c.endswith("year_outcome")]
print("outcome columns:   ", outcome_cols)
print("prediction columns:", kfre_cols)
print()
print(df[outcome_cols + kfre_cols].isnull().sum())

## 8. Discrimination Performance

`bootstrap_metric_ci` returns a point estimate with a percentile bootstrap confidence
interval. `plot_kfre_metrics` draws ROC and precision-recall curves for every model and
horizon combination.

Two cautions when reading the table.

**Threshold-dependent metrics use a 0.5 cut-point.** Against a 2-year event rate below
1%, a 0.5 threshold sits far from any clinically meaningful operating point. Precision,
sensitivity, and specificity below are descriptive only. The thresholds used in the CKD
referral literature are 3%, 5%, and 15% predicted 5-year risk.

**AUC ROC is not a C-statistic.** Published KFRE C-statistics are Harrell's C on censored
time-to-event data. The values below are binary ROC AUC at a fixed horizon. Any gap
between the two reflects a difference in estimand.

In [ ]:
metric_labels = {
    "precision": "Precision/PPV",
    "average_precision": "Average Precision",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "auc_roc": "AUC ROC",
    "brier": "Brier Score",
}
metrics = list(metric_labels)

records = []
for n in (4, 6):
    for y in (2, 5):
        col = f"{y}_year_{n}_var_kfre"
        y_true = df[f"esrd_{y}_year_outcome"]
        y_score = df[f"kfre_{n}var_{y}year"]
        for m in metrics:
            r = bootstrap_metric_ci(
                y_true, y_score, metric=m, n_boot=1000, seed=42, progress=True
            )
            records.append(
                {
                    "Outcome Metrics": metric_labels[m],
                    "column": col,
                    "cell": f"{r['point']:.3f} ({r['lower']:.3f}\u2013{r['upper']:.3f})",
                }
            )

col_order = [f"{y}_year_{n}_var_kfre" for n in (4, 6) for y in (2, 5)]
ci_table = (
    pd.DataFrame(records)
    .pivot(index="Outcome Metrics", columns="column", values="cell")
    .reindex([metric_labels[m] for m in metrics])
    .reindex(columns=col_order)
)
ci_table

In [ ]:
plot_kfre_metrics(
    df=df,
    num_vars=[4, 6],
    figsize=[6, 6],
    mode="plot",
    image_prefix="performance",
    bbox_inches="tight",
    plot_type="all_plots",
    show_years=[2, 5],
    plot_combinations=True,
    show_subplots=True,
    decimal_places=3,
    save_plots=True,
    image_path_png="../images",
    image_filename="../images/leicester_kfre_performance.png",
)

### 4-variable versus 6-variable

Discrimination is effectively identical between the two forms at both horizons. This is
consistent with the parsimony argument made by Tangri et al., who recommended the
4-variable equation for clinical implementation on the grounds that the additional
covariates in the 6- and 8-variable forms bought little extra performance.

## 9. CKD Stage Distribution and Risk Figures

`class_ckd_stages` assigns KDIGO GFR-category labels from eGFR. The cohort inclusion
criterion for this dataset was two CKD-EPI eGFR values below 60 ml/min/1.73 m<sup>2</sup>,
so the observed range spans stages G3a through G5 with the large majority in G3.

In [ ]:
df = class_ckd_stages(
    df=df, egfr_col="epi_egfr", stage_col="stage", combined_stage_col="stage_combined"
)

# Order stages by descending median eGFR so the axis reads G3a -> G5
# regardless of the exact label strings the package emits.
stage_order = (
    df.groupby("stage")["epi_egfr"].median().sort_values(ascending=False).index.tolist()
)
df["stage"].value_counts().reindex(stage_order)

In [ ]:
# Predicted 2-year risk by CKD stage, all stages present in the cohort.
data_by_stage = [
    df.loc[df["stage"] == s, "kfre_4var_2year"].dropna() for s in stage_order
]
tick_labels = [
    f"{s.replace('CKD Stage ', 'Stage ')}\n(n={len(d):,})"
    for s, d in zip(stage_order, data_by_stage)
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(data_by_stage, tick_labels=tick_labels, showfliers=False)
ax.set_yscale("log")
ax.set_xlabel("CKD Stage")
ax.set_ylabel("Predicted 2-year risk (log scale)")
ax.set_title("KFRE 4-Variable, 2-Year Risk by CKD Stage")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../images/leicester_risk_by_stage.png", dpi=300)
plt.show()

In [ ]:
# 4-variable versus 6-variable agreement at 2 years.
# Log-log axes because predicted risk spans several orders of magnitude.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    df["kfre_4var_2year"], df["kfre_6var_2year"], alpha=0.2, s=6, edgecolors="none"
)

lo = min(df["kfre_4var_2year"].min(), df["kfre_6var_2year"].min())
hi = max(df["kfre_4var_2year"].max(), df["kfre_6var_2year"].max())
ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1, label="Identity")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("4-Variable KFRE, 2-year risk")
ax.set_ylabel("6-Variable KFRE, 2-year risk")
ax.set_title("4- vs. 6-Variable KFRE: 2-Year Risk Estimates")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../images/leicester_4var_vs_6var.png", dpi=300)
plt.show()

print(
    "Spearman correlation (2-year): "
    f"{df['kfre_4var_2year'].corr(df['kfre_6var_2year'], method='spearman'):.4f}"
)

## 10. Summary

This notebook demonstrates a complete `kfre` workflow on a real cohort:

1. Convert urine ACR to mg/g before scoring. This is the most common source of error when
   applying the KFRE to non-US data, and it silently corrupts calibration while leaving
   discrimination untouched.
2. Exclude records the equation cannot score (`ln(ACR)` undefined at zero) and report the
   count.
3. Label outcomes at fixed horizons with `class_esrd_outcome`, and check sensitivity to
   the censoring assumption with `censor_incomplete=True`.
4. Compute 4- and 6-variable risk at 2 and 5 years in one call with
   `add_kfre_risk_col`, using the non-North American baseline where appropriate.
5. Assess discrimination with bootstrap confidence intervals, keeping in mind that
   threshold metrics at a 0.5 cut-point are uninformative for a rare outcome.

### Further work

Discrimination alone does not establish that a risk equation is fit for use in a given
population. Calibration is the substantive next step: calibration slope and intercept,
observed-to-expected ratio, calibration plots by risk decile, and decision curve
analysis at the 3%, 5%, and 15% thresholds used in CKD referral guidance.